In [36]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [37]:
# Load dataset
df = pd.read_csv("spam.csv", encoding="latin-1")[['Category','Message']]

In [38]:
# Convert labels: ham=0, spam=1
df['label'] = df['Category'].map({'ham':0, 'spam':1})

In [39]:
# Features and labels
X_texts = df['Message'].values
y = df['label'].values

In [40]:
# TF-IDF vectorization
vectorizer = TfidfVectorizer(stop_words='english', max_features=3000)  # limit vocab size
X = vectorizer.fit_transform(X_texts).toarray()

In [41]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [42]:
# ANN model
model = Sequential()
model.add(Dense(64, input_dim=X.shape[1], activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [43]:
# Train
model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=1)

Epoch 1/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8795 - loss: 0.3436
Epoch 2/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9807 - loss: 0.0849
Epoch 3/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9935 - loss: 0.0248
Epoch 4/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9984 - loss: 0.0103
Epoch 5/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9993 - loss: 0.0058
Epoch 6/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9991 - loss: 0.0041
Epoch 7/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9991 - loss: 0.0036
Epoch 8/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9991 - loss: 0.0032
Epoch 9/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9996 - loss: 0.0026  
Epoch 10/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9996 - loss: 0.0022


In [44]:
# Evaluate
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print("Test Accuracy:", acc)

Test Accuracy: 0.9901345372200012


In [45]:
# Predict on new messages
new_msgs = [
    "Congratulations! You have won a free ticket",
    "Meeting tomorrow at 10 AM with the project team"
]

In [46]:
X_new = vectorizer.transform(new_msgs).toarray()
preds = (model.predict(X_new) > 0.5).astype("int32")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step


In [47]:
for msg, pred in zip(new_msgs, preds):
    print(f"Message: {msg} → Prediction: {'Spam' if pred==1 else 'Ham'}")

Message: Congratulations! You have won a free ticket → Prediction: Spam
Message: Meeting tomorrow at 10 AM with the project team → Prediction: Ham


In [ ]:
import gradio as gr

# Define prediction function
def predict_message(msg):
    # Transform input using the same vectorizer
    X_new = vectorizer.transform([msg]).toarray()
    pred = (model.predict(X_new) > 0.5).astype("int32")[0][0]
    return "Spam" if pred == 1 else "Ham"

# Create Gradio interface
iface = gr.Interface(
    fn=predict_message,
    inputs=gr.Textbox(lines=2, placeholder="Enter a message..."),
    outputs="text",
    title="Spam Detector",
    description="Type a message and see if it's Spam or Ham"
)

# Launch interface
iface.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
